# Week 5: Hospital General Information (Bronze, Silver, Gold join)

Purpose: add the second CMS dataset from the proposal (LO1), so H2 (quality), H3 (ownership) and the Week 6 regression can run.

Source: CMS Provider Data Catalog, *Hospital General Information* (dataset xubh-q36u), archived snapshot of 2024-10-30, file `Hospital_General_Information_2024-10-30.csv`. 5,394 hospitals x 39 columns. Join key: `Facility ID`, the 6-character CCN.

Outputs: `workspace.cms.bronze_hospital_info`, `workspace.cms.silver_hospital_info`, `workspace.cms.gold_dim_hospital_profile` (one row per inpatient hospital, joined to Gold on `ccn`).

Decisions (2026-11-02, revised 2026-11-03; logged in docs/data_quality_log.md)

1. Snapshot timing: use the CMS archived snapshot of 2024-10-30, inside the CY2024 claims year, so ratings, ownership and CCNs describe hospitals as they were when the claims were paid (DQ-24). The 2026 file was replaced on 2026-11-03 because 17 hospitals had changed CCN after converting to Critical Access, Rural Emergency or psychiatric status.

2. "Not Available" star ratings: kept as their own "Unrated" category. A numeric rating column (NULL for Unrated) is stored next to it so rated hospitals can still be ranked.

3. Ownership: 3 groups. For-profit = Proprietary + Physician. Non-profit = Voluntary non-profit (Private, Other, Church). Government = Local, Hospital District or Authority, State, Federal, Tribal. The original CMS value is kept in Silver.

4. Unmatched inpatient hospitals (7 expected): kept with NULL attributes and an `is_profile_matched` flag; excluded only from ownership and quality tests (DQ-25).

5. Gold design: a separate one-row-per-hospital table, `gold_dim_hospital_profile`, built here. `gold_dim_hospital` (notebook 03) is not modified, so every notebook stays rerunnable on its own.

In [0]:
# ---------------------------------------------------------------
# Cell 2: Config
# Purpose: every path, table name and expected value in one place
# Inputs: none | Outputs: Python variables used by later cells
# Expected values come from local profiling of the 2024-10-30 snapshot (2026-11-03)
# ---------------------------------------------------------------
CATALOG = "workspace"
SCHEMA = "cms"
VOLUME = "raw"
FILE_NAME = "Hospital_General_Information_2024-10-30.csv"

SOURCE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{FILE_NAME}"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_hospital_info"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_hospital_info"

EXPECTED_ROWS = 5_394
EXPECTED_SOURCE_COLS = 39

print("Source:", SOURCE_PATH)
print("Target:", BRONZE_TABLE)

In [0]:
# ---------------------------------------------------------------
# Cell 3: Confirm the file landed in the volume
# Purpose: fail early if the CSV was not uploaded
# Inputs: /Volumes/workspace/cms/raw/ | Outputs: file listing
# Expected: 3 files (both Hospital General Information files and the inpatient CSV);
#           the 2024-10-30 file must be among them
# ---------------------------------------------------------------
files = dbutils.fs.ls(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/")
display(files)
assert any(f.name == FILE_NAME for f in files), f"{FILE_NAME} not found in the volume"

In [0]:
# ---------------------------------------------------------------
# Cell 4: Bronze ingest
# Purpose: land the CMS file as-is, every column STRING, plus lineage
# Inputs: SOURCE_PATH | Outputs: workspace.cms.bronze_hospital_info
# Expected: 5,394 rows, 39 source columns, 0 corrupt records
# Same pattern as 01_bronze_ingest.py
# ---------------------------------------------------------------
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# Step A: read only the header row to get the column names exactly as CMS wrote them
header_cols = spark.read.option("header", True).csv(SOURCE_PATH).columns
assert len(header_cols) == EXPECTED_SOURCE_COLS, f"Expected {EXPECTED_SOURCE_COLS} columns, got {len(header_cols)}"

# Step B: explicit all-string schema, so Facility ID keeps its leading zeros
# and any malformed line lands in _corrupt_record instead of being dropped
bronze_schema = StructType(
    [StructField(c, StringType(), True) for c in header_cols]
    + [StructField("_corrupt_record", StringType(), True)]
)

raw_df = (
    spark.read
    .schema(bronze_schema)
    .option("header", True)
    .option("enforceSchema", False)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", False)
    .option("encoding", "UTF-8")
    .csv(SOURCE_PATH)
)

# Step C: add lineage columns, the only things Bronze adds
bronze_df = (
    raw_df
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

# Write with Delta column mapping so the CMS column names (spaces, /) are kept as-is
bronze_df.createOrReplaceTempView("bronze_hospital_info_src")
spark.sql(f"CREATE OR REPLACE TABLE {BRONZE_TABLE} TBLPROPERTIES ('delta.columnMapping.mode' = 'name') AS SELECT * FROM bronze_hospital_info_src")

spark.sql(f"""
COMMENT ON TABLE {BRONZE_TABLE} IS
'Bronze: CMS Hospital General Information (Provider Data Catalog xubh-q36u, snapshot 2024-10-30, archived), loaded as-is. All source columns are STRING. One row = one hospital (Facility ID = CCN). Lineage: _ingest_ts, _source_file.'
""")

# Step D: row count and corrupt records
bronze = spark.table(BRONZE_TABLE)
n_rows = bronze.count()
n_corrupt = bronze.filter(F.col("_corrupt_record").isNotNull()).count()
print("Rows:", n_rows, "| Corrupt records:", n_corrupt)
assert n_rows == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS} rows, got {n_rows}"
assert n_corrupt == 0, f"{n_corrupt} corrupt records"
print("Wrote", BRONZE_TABLE)

In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 5: Bronze checks
--- Purpose: confirm the key column is complete, unique and 6 characters
--- Inputs: workspace.cms.bronze_hospital_info | Outputs: one result row
--- Expected: 5,394 rows; 5,394 distinct Facility IDs; 0 NULL IDs; 0 IDs not 6 characters
--- ---------------------------------------------------------------

SELECT 
    COUNT(*) AS n_rows,
    COUNT (DISTINCT `Facility ID`) AS n_distinct_ids,
    SUM(CASE WHEN `Facility ID` IS NULL THEN 1 ELSE 0 END) AS n_null_ids,
    SUM(CASE WHEN LENGTH(`Facility ID`) != 6 THEN 1 ELSE 0 END) AS n_not_6_chars

FROM    
    workspace.cms.bronze_hospital_info;


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 6a: Hospital Type profile
--- Purpose: see which hospital types exist; the inpatient file should match only one
--- Inputs: workspace.cms.bronze_hospital_info | Outputs: one row per type
--- Expected: 6 types; Acute Care Hospitals 3,153 is the largest
--- ---------------------------------------------------------------

SELECT `Hospital Type`, COUNT(*) AS n_hospitals
FROM workspace.cms.bronze_hospital_info
GROUP BY `Hospital Type`
ORDER BY n_hospitals DESC;


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 6b: Hospital Ownership profile
--- Purpose: every value must map to an ownership group in Silver (decision 3)
--- Inputs: workspace.cms.bronze_hospital_info | Outputs: one row per ownership value
--- Expected: 12 values; Voluntary non-profit - Private 2,277 is the largest
--- ---------------------------------------------------------------

SELECT `Hospital Ownership`, COUNT(*) AS n_hospitals
FROM workspace.cms.bronze_hospital_info
GROUP BY `Hospital Ownership`
ORDER BY n_hospitals DESC;


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 6c: Hospital overall rating profile
--- Purpose: confirm the rating values before typing them in Silver (decision 2)
--- Inputs: workspace.cms.bronze_hospital_info | Outputs: one row per rating value
--- Expected: 6 values: 1 to 5 plus "Not Available" (2,568)
--- ---------------------------------------------------------------

SELECT `Hospital overall rating`, COUNT(*) AS n_hospitals
FROM workspace.cms.bronze_hospital_info
GROUP BY `Hospital overall rating`
ORDER BY `Hospital overall rating`;


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 7: Join preview
--- Purpose: how many inpatient hospitals find a match before Silver is built
--- Inputs: workspace.cms.gold_dim_hospital, workspace.cms.bronze_hospital_info | Outputs: counts
--- Expected: 2,906 inpatient hospitals; 2,899 matched; 7 unmatched
--- Silver ccn is already 6 digits (DQ-02), so no LPAD is needed
--- ---------------------------------------------------------------

SELECT 
    COUNT(*) AS n_inpatient_hospitals,
    COUNT (b.`Facility ID`) AS n_matched,
    COUNT (*) - COUNT (b.`Facility ID`) AS n_unmatched

FROM workspace.cms.gold_dim_hospital h 
LEFT JOIN workspace.cms.bronze_hospital_info b 
    ON h.ccn = b.`Facility ID`;


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 7b: Unmatched inpatient hospitals
--- Purpose: list the hospitals with no Hospital General Information row, to explain them in DQ-25
--- Inputs: workspace.cms.gold_dim_hospital, workspace.cms.bronze_hospital_info | Outputs: one row per unmatched hospital
--- Expected: 7 rows
--- ---------------------------------------------------------------

SELECT 
    h.ccn, 
    h.provider_name, 
    h.provider_city, 
    h.state_abbr

FROM 
    workspace.cms.gold_dim_hospital h

LEFT JOIN 
    workspace.cms.bronze_hospital_info b 
    ON h.ccn = b.`Facility ID`

WHERE
    b.`Facility ID` IS NULL 

ORDER BY 
    h.state_abbr, h.provider_name;


## Silver

Next: type the rating (numeric column, NULL for Unrated, plus a rating category with "Unrated"), map ownership to the three groups (plus Federal (VA/DoD) for the two federal values that have no inpatient claims), keep the original CMS columns for traceability, and add checks.